<a href="https://colab.research.google.com/github/Woradaaa/BDA_project/blob/main/BDA_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mock data for Top 10 Global Gacha Games in 2026
gacha_data = """Game Name: Genshin Impact
Developer: HoYoverse
Genre: Open-World Action RPG
Key Features: Massive open-world exploration, elemental combat system, deep lore
Available Platforms: PC, PS4, PS5, iOS, Android
--------------------------------------------------
Game Name: Honkai: Star Rail
Developer: HoYoverse
Genre: Turn-Based Strategy RPG
Key Features: Strategic team-building, interstellar adventure, high-quality animations
Available Platforms: PC, PS5, iOS, Android
--------------------------------------------------
Game Name: Wuthering Waves
Developer: Kuro Games
Genre: Open-World Action RPG
Key Features: Fast-paced fluid combat, monster echo collection system, parkour movement
Available Platforms: PC, PS5, iOS, Android
--------------------------------------------------
Game Name: Zenless Zone Zero
Developer: HoYoverse
Genre: Urban Fantasy Action RPG
Key Features: Stylish fast-paced combat, rogue-like hollow exploration, retro-futuristic aesthetic
Available Platforms: PC, PS5, iOS, Android
--------------------------------------------------
Game Name: Love and Deepspace
Developer: Infold Games
Genre: 3D Immersive Romance / Action RPG
Key Features: First-person perspective storyline, 3D character interactions, real-time action combat
Available Platforms: iOS, Android
--------------------------------------------------
Game Name: Goddess of Victory: Nikke
Developer: Shift Up
Genre: Third-Person Shooter / Idle RPG
Key Features: Cover-based shooting mechanics, dark sci-fi narrative, idle progression
Available Platforms: PC, iOS, Android
--------------------------------------------------
Game Name: Pokémon TCG Pocket
Developer: Creatures Inc. / DeNA
Genre: Digital Collectible Card Game
Key Features: Immersive 3D cards, quick simplified battle format, daily free pack pulls
Available Platforms: iOS, Android
--------------------------------------------------
Game Name: Arknights
Developer: Hypergryph
Genre: Strategy Tower Defense
Key Features: Strategic grid-based deployments, deep industrial sci-fi world-building, base management
Available Platforms: PC, iOS, Android
--------------------------------------------------
Game Name: Infinity Nikki
Developer: Infold Games
Genre: Open-World Dress-Up Adventure
Key Features: Exploration through fashion, cozy puzzle-solving, vibrant open-world platforming
Available Platforms: PC, PS5, iOS, Android
--------------------------------------------------
Game Name: Fate/Grand Order
Developer: Lasengle
Genre: Turn-Based RPG
Key Features: Massive visual novel storytelling, historical and mythical hero collection, iconic Fate franchise lore
Available Platforms: iOS, Android
"""

# Define the file name
file_name = "10_gacha_games_mock_data.txt"

# Create and write the mock data to the text file
with open(file_name, "w", encoding="utf-8") as file:
    file.write(gacha_data)

print(f"✅ File '{file_name}' has been successfully created containing 10 gacha games!")

# Trigger the download to your local machine
from google.colab import files
files.download(file_name)

✅ File '10_gacha_games_mock_data.txt' has been successfully created containing 10 gacha games!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Install Required Libraries RAG, Local LLM และ Gradio UI
!pip install -q langchain "langchain-community<0.3.28" langchain-huggingface faiss-cpu sentence-transformers
!pip install -q transformers accelerate bitsandbytes gradio
!pip install gradio -q

In [ ]:
import torch
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFacePipeline
# ✅ FIX: BitsAndBytesConfig is now imported below!
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 1. Load the .txt file
file_name = "10_gacha_games_mock_data.txt"
loader = TextLoader(file_name, encoding="utf-8")
docs = loader.load()

# 2. Split the text into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n--------------------------------------------------\n", "\n\n", "\n", " "]
)
splits = text_splitter.split_documents(docs)

# 3. Create HuggingFace Embeddings and FAISS Vector Store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splits, embeddings)

# Set up the vector store as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4. Load OpenThaiGPT-1.0.0-7b-chat with 4-bit quantization
model_id = "openthaigpt/openthaigpt-1.0.0-7b-chat"

# Set up the new quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model using the config object
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

# Create a text-generation pipeline
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.1
)
llm = HuggingFacePipeline(pipeline=text_pipeline)

# 5. Build the Retrieval Chain
system_prompt = (
    "You are an enthusiastic and knowledgeable Sneakerhead and Brand Guide. "
    "Your job is to answer questions about the top global sneaker brands based on the provided context. "
    "Only use the information given in the context to answer the user's question. "
    "If you don't know the answer, politely say that you don't have that information. "
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Create the chains
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# --- Test the System ---
print("RAG System Initialized! Asking a test question...\n")

query = "Can you recommend some gacha game that are good for studying and what should I order?"
response = rag_chain.invoke({"input": query})

print(f"User Question: {query}")
print(f"MFU Cafe Guide: {response['answer']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG System Initialized! Asking a test question...

User Question: Can you recommend some gacha game that are good for studying and what should I order?
MFU Cafe Guide: System: You are an enthusiastic and helpful Gacha Game Expert.Your job is to recommend top global gacha games based on the provided context.Only use the information given in the context to answer the user's question.If you don't know the answer, politely say that you don't have that information.

Context:
Game Name: Genshin Impact
Developer: HoYoverse
Genre: Open-World Action RPG
Key Features: Massive open-world exploration, elemental combat system, deep lore
Available Platforms: PC, PS4, PS5, iOS, Android
--------------------------------------------------
Game Name: Honkai: Star Rail
Developer: HoYoverse
Genre: Turn-Based Strategy RPG
Key Features: Strategic team-building, interstellar adventure, high-quality animations
Available Platforms: PC, PS5, iOS, Android

--------------------------------------------------
Game N

In [ ]:
import gradio as gr

# This function connects Gradio's chat input to your LangChain RAG pipeline
def chat_with_gacha_guide(message, history):
    # Pass the user's message into the existing RAG chain
    response = rag_chain.invoke({"input": message})

    # Extract and return just the answer text from the response dictionary
    return response['answer']

# Create the ChatInterface
demo = gr.ChatInterface(
    fn=chat_with_gacha_guide,
    title="TOP 10 GLOBAL GACHA GAMES (2026)",
    description="Ask me anything about the top gacha games in the world! (e.g., 'What are some good open-world action RPGs?' or 'Who developed Zenless Zone Zero?')",
    examples=[
        "Can you recommend a game with turn-based strategy?",
        "Which games are available to play on PS5?",
        "What are the key features of Wuthering Waves?"
    ],
    theme="soft"
)

# Launch the app and create a public link
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77c29dc73dfbbea798.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
